In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

/home/bingxing2/ailab/scxlab0179/.conda/envs/bio/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/bingxing2/ailab/scxlab0179/.conda/envs/bio/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [2]:
import os
os.chdir("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_brain_simo/")

In [3]:
# 设置 Matplotlib 的参数
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42  # 确保文本为矢量
mpl.rcParams['ps.fonttype'] = 42

In [4]:
rna = sc.read_h5ad("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_brain_simo/rna.h5ad")
methy = sc.read_h5ad("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_brain_simo/met.h5ad")
st = sc.read_h5ad("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_brain_simo/spatial.h5ad")

In [5]:
rna

AnnData object with n_obs × n_vars = 55803 × 25299
    obs: 'cluster', 'subcluster', 'cell_type', 'cell_subtype', 'domain', 'protocol', 'dataset'
    var: 'chrom', 'chromStart', 'chromEnd', 'name', 'score', 'strand', 'thickStart', 'thickEnd', 'itemRgb', 'blockCount', 'blockSizes', 'blockStarts', 'gene_id', 'gene_type', 'mgi_id', 'havana_gene', 'tag', 'genome', 'n_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg'

In [6]:
methy

AnnData object with n_obs × n_vars = 3377 × 79796
    obs: 'cell_type', 'domain', 'protocol', 'dataset'
    var: 'geneID', 'chrom', 'chromStart', 'chromEnd', 'name', 'score', 'strand', 'thickStart', 'thickEnd', 'itemRgb', 'blockCount', 'blockSizes', 'blockStarts', 'gene_type', 'gene_status', 'gene_name', 'havana_gene', 'tag', 'genome'
    layers: 'norm'

In [7]:
mch = methy.var_names.str.endswith('_mCH')
mcg = methy.var_names.str.endswith('_mCG')

In [8]:
methy_mch = methy[:, mch].copy()
methy_mcg = methy[:, mcg].copy()

In [9]:
methy_mch.var_names = methy_mch.var_names.str.split('_mCH').str[0]
methy_mcg.var_names = methy_mcg.var_names.str.split('_mCG').str[0]

In [10]:
# df = pd.read_csv("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_brain_simo/mCG_genebody_mouse.txt",sep="\t")

In [11]:
locs = pd.read_csv("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_brain_simo/pos.csv")
locs = locs.loc[:,['x','y']].to_numpy()
st.obsm['spatial'] = locs.copy()
st

AnnData object with n_obs × n_vars = 1075 × 31053
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'seurat_clusters'
    var: 'features'
    obsm: 'spatial'

In [12]:
rna.layers['counts'] = rna.X.copy()
sc.pp.normalize_total(rna)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna,n_top_genes=3000) #batch_key="batch", , min_mean=0.02, max_mean=4, min_disp=0.5
genelist = rna.var.index[rna.var['highly_variable']==True]

In [13]:
sc.pp.highly_variable_genes(methy_mcg,layer="norm",n_top_genes=3000) #, min_mean=0.05, max_mean=1.5, min_disp=.5,, batch_key="batch"
peaklist = methy_mcg.var.index[methy_mcg.var['highly_variable']==True]
sc.pp.highly_variable_genes(methy_mch,layer="norm",n_top_genes=3000) #, min_mean=0.05, max_mean=1.5, min_disp=.5,, batch_key="batch"
peaklist2 = methy_mch.var.index[methy_mch.var['highly_variable']==True]

In [14]:
sc.pp.normalize_total(st)
sc.pp.log1p(st)
sc.pp.highly_variable_genes(st,n_top_genes=3000) #batch_key="batch", , min_mean=0.02, max_mean=4, min_disp=0.5
stlist = st.var.index[st.var['highly_variable']==True]

In [15]:
features = list(set(genelist) | set(peaklist) | set(stlist))
rna_feat = list(set(rna.var_names)&set(features))
methy_feat = list(set(methy_mcg.var_names)&set(features))
st_feat = list(set(st.var_names)&set(features))

In [16]:
print(len(rna_feat))
print(len(methy_feat))
print(len(st_feat))
print(len(features))

6428
7774
6769
8028


In [17]:
rna.layers['normalized'] = rna.X.copy()
methy_mcg.layers['normalized'] = methy_mcg.X.copy()
st.layers['normalized'] = st.X.copy()
rna.layers['norm'] = rna.layers['normalized'].copy()
st.layers['norm'] = st.layers['normalized'].copy()
methy_mch.layers['normalized'] = methy_mch.X.copy()
sc.pp.scale(methy_mch)
sc.pp.scale(rna)
sc.pp.scale(methy_mcg)
sc.pp.scale(st)

/home/bingxing2/ailab/scxlab0179/.conda/envs/bio/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
/home/bingxing2/ailab/scxlab0179/.conda/envs/bio/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


In [18]:
import anndata as ad
adata = ad.concat([rna[:,rna_feat],methy_mcg[:,methy_feat],st[:,st_feat]],axis="obs",
                  join="outer",label="modality",keys=["rna","methy","st"])
adata.uns['rna_feat'] = rna_feat
adata.uns['methy_feat'] = methy_feat
adata.uns['st_feat'] = st_feat
adata.uns['rna_hvg'] = list(set(genelist) & set(adata.var_names))
adata.uns['methy_hvg'] = list(set(peaklist) & set(adata.var_names))
adata.uns['st_hvg'] = list(set(stlist) & set(adata.var_names))

/home/bingxing2/ailab/scxlab0179/.conda/envs/bio/lib/python3.11/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(


In [19]:
adata.write("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_brain_simo/merged_mcg.h5ad")

In [20]:
features = list(set(genelist) | set(peaklist2) | set(stlist))
rna_feat = list(set(rna.var_names)&set(features))
methy_feat = list(set(methy_mch.var_names)&set(features))
st_feat = list(set(st.var_names)&set(features))
import anndata as ad
adata = ad.concat([rna[:,rna_feat],methy_mch[:,methy_feat],st[:,st_feat]],axis="obs",
                  join="outer",label="modality",keys=["rna","methy","st"])
adata.uns['rna_feat'] = rna_feat
adata.uns['methy_feat'] = methy_feat
adata.uns['st_feat'] = st_feat
adata.uns['rna_hvg'] = list(set(genelist) & set(adata.var_names))
adata.uns['methy_hvg'] = list(set(peaklist) & set(adata.var_names))
adata.uns['st_hvg'] = list(set(stlist) & set(adata.var_names))

/home/bingxing2/ailab/scxlab0179/.conda/envs/bio/lib/python3.11/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(


In [21]:
adata.write("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_brain_simo/merged_mch.h5ad")

In [22]:
features = list(set(genelist) | set(peaklist) | set(peaklist2) | set(stlist))
rna_feat = list(set(rna.var_names)&set(features))
methy_feat = list(set(methy_mcg.var_names)&set(features))
methy_feat2 = list(set(methy_mch.var_names)&set(features))
st_feat = list(set(st.var_names)&set(features))
import anndata as ad
adata = ad.concat([rna[:,rna_feat],methy_mcg[:,methy_feat],methy_mch[:,methy_feat2],st[:,st_feat]],axis="obs",
                  join="outer",label="modality",keys=["rna","methy_mCG","methy_mCH","st"])
adata.uns['rna_feat'] = rna_feat
adata.uns['methy_mcg_feat'] = methy_feat
adata.uns['methy_mch_feat'] = methy_feat2
adata.uns['st_feat'] = st_feat
adata.uns['rna_hvg'] = list(set(genelist) & set(adata.var_names))
adata.uns['methy_mcg_hvg'] = list(set(peaklist) & set(adata.var_names))
adata.uns['methy_mch_hvg'] = list(set(peaklist2) & set(adata.var_names))
adata.uns['st_hvg'] = list(set(stlist) & set(adata.var_names))

/home/bingxing2/ailab/scxlab0179/.conda/envs/bio/lib/python3.11/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(
/home/bingxing2/ailab/scxlab0179/.conda/envs/bio/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [23]:
print(len(rna_feat))
print(len(methy_feat))
print(len(methy_feat2))
print(len(st_feat))
print(len(features))

8107
10019
9773
8557
10275


In [24]:
shared_feat = list(set(rna_feat)&set(methy_feat)&set(methy_feat2)&set(st_feat))
print(len(shared_feat))

7218


In [27]:
adata.write("/home/bingxing2/ailab/group/ai4bio/sunjianle/mouse_brain_simo/merged.h5ad")